# Zeitreihenanalyse globaler Kraftstoffpreise (2020–2026)

**Länder:** Deutschland, Frankreich, Italien, Indonesien
**Zielvariable:** `petrol_usd_liter` (Benzinpreis in USD/Liter, wöchentlich)

## Methodischer Aufbau

| Block | Inhalt |
|---|---|
| 1 | Exploratory Data Analysis (EDA) |
| 2 | Stationaritätstests (ADF, PP, KPSS) & Transformationen |
| 3 | Train/Test-Split (70/30, temporal) |
| 4 | Strukturbruch-Analyse (Chow, QLR) |
| 5 | Univariate Modellierung: ARIMA (Box–Jenkins) + TS-Cross-Validation |
| 6 | Multivariate Modellierung: VAR + Granger-Kausalität |
| 7 | Modellselektion: AIC/BIC/HQ + Residualdiagnostik |
| 8 | State-Space-Modell (Unobserved Components) |
| 9 | Prognose + 95%-Intervalle + Evaluation (MSE/RMSE/MAE) |

## Wichtige Vorbehalte

1. **Indonesien ist subventioniert** (`subsidy_level=High`) – Preise reagieren strukturell anders auf den Brent-Preis. Ergebnisse für ID sind nicht direkt vergleichbar mit DE/FR/IT.
2. **`tax_percentage` schwankt wöchentlich unrealistisch stark** (synthetische Komponente). Wird zwar im VAR mitmodelliert, aber Interpretation mit Vorsicht.
3. **Pessimistic Bias:** Das Modell wird auf Daten bis ~Mitte 2024 trainiert und auf 2024–2026 getestet. Da diese Periode überdurchschnittliche Volatilität enthält (Brent springt auf 130 USD), sind die Out-of-Sample-Fehler systematisch nach oben verzerrt.

## 0 · Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Statistische Tests & Modelle
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.structural import UnobservedComponents
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Setup abgeschlossen.')

## 1 · Daten laden

Vier CSV-Dateien, eine pro Land, jeweils 327 wöchentliche Beobachtungen.

In [ ]:
DATA_DIR = Path('.')  # ggf. anpassen, falls Notebook nicht im selben Ordner wie die CSVs liegt

FILES = {
    'Germany':    'fuel_prices_Germany.csv',
    'France':     'fuel_prices_France.csv',
    'Italy':      'fuel_prices_Italy.csv',
    'Indonesia':  'fuel_prices_Indonesia.csv',
}

def load_country(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['date'])
    df = df.sort_values('date').set_index('date')
    # Wöchentliche Frequenz explizit setzen (W-MON, da die Daten montags beginnen)
    df = df.asfreq('W-MON')
    return df

data = {country: load_country(DATA_DIR / fname) for country, fname in FILES.items()}

for c, df in data.items():
    print(f'{c:12s} | {len(df)} Beobachtungen | {df.index.min().date()} → {df.index.max().date()}')

data['Germany'].head()

## 2 · Exploratory Data Analysis

### 2.1 Deskriptive Statistik

Zentrale Kennzahlen für jedes Land: Mittelwert, Standardabweichung, Min/Max, Schiefe, Kurtosis.

In [ ]:
TARGET = 'petrol_usd_liter'
NUMERIC_COLS = ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter',
                'brent_crude_usd', 'tax_percentage']

summary_rows = []
for country, df in data.items():
    s = df[TARGET]
    summary_rows.append({
        'Country': country,
        'N': len(s),
        'Mean': s.mean(),
        'Std': s.std(),
        'Min': s.min(),
        'Max': s.max(),
        'Skew': s.skew(),
        'Kurtosis': s.kurtosis(),
        # Coefficient of Variation – Maß für relative Streuung
        'CV': s.std() / s.mean(),
    })
summary = pd.DataFrame(summary_rows).set_index('Country')
summary

### 2.2 Lumpy / Erratic Pattern Detection

Nach Syntetos & Boylan (2005) wird ein Zeitreihenmuster über zwei Kennzahlen klassifiziert:

- **ADI** (Average Demand Interval): Durchschnittlicher Abstand zwischen Beobachtungen mit Bewegung
- **CV²** (Squared Coefficient of Variation): Streuungsmaß der Differenzen

Schwellwerte: ADI > 1.32 → intermittierend; CV² > 0.49 → erratisch.
Da unsere Preisdaten kontinuierlich sind, erwarten wir ADI ≈ 1 (kein intermittentes Muster), aber CV² ist interessant für die Volatilität.

In [ ]:
def adi_cv2(series: pd.Series) -> dict:
    diffs = series.diff().dropna()
    # ADI: 1 / Anteil der Wochen mit Preisänderung
    nonzero = (diffs != 0).sum()
    adi = len(diffs) / nonzero if nonzero > 0 else np.inf
    # CV² der Erstdifferenzen
    cv2 = (diffs.std() / abs(diffs.mean()))**2 if diffs.mean() != 0 else np.nan
    return {'ADI': adi, 'CV2': cv2}

pattern = pd.DataFrame({c: adi_cv2(data[c][TARGET]) for c in data}).T
def classify(row):
    if row['ADI'] < 1.32 and row['CV2'] < 0.49: return 'Smooth'
    if row['ADI'] >= 1.32 and row['CV2'] < 0.49: return 'Intermittent'
    if row['ADI'] < 1.32 and row['CV2'] >= 0.49: return 'Erratic'
    return 'Lumpy'
pattern['Class'] = pattern.apply(classify, axis=1)
pattern

### 2.3 Linienplots – Petrol, Diesel, LPG, Brent

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
for ax, country in zip(axes, data):
    df = data[country]
    ax.plot(df.index, df['petrol_usd_liter'], label='Petrol', linewidth=1.4)
    ax.plot(df.index, df['diesel_usd_liter'], label='Diesel', linewidth=1.0, alpha=0.8)
    ax.plot(df.index, df['lpg_usd_liter'],    label='LPG',    linewidth=1.0, alpha=0.6)
    ax2 = ax.twinx()
    ax2.plot(df.index, df['brent_crude_usd'], color='black', linestyle=':', alpha=0.5, label='Brent (USD)')
    ax.set_title(f'{country} – Kraftstoffpreise (USD/L)')
    ax.set_ylabel('USD/Liter')
    ax2.set_ylabel('Brent USD')
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 2.4 Histogramme und Boxplots zur Ausreißerdiagnose

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for j, country in enumerate(data):
    s = data[country][TARGET]
    axes[0, j].hist(s, bins=30, edgecolor='black')
    axes[0, j].set_title(f'{country} – Histogramm')
    axes[0, j].set_xlabel('USD/L')
    axes[1, j].boxplot(s, vert=True)
    axes[1, j].set_title(f'{country} – Boxplot')
    axes[1, j].set_xticks([])
plt.tight_layout()
plt.show()

**Lesehilfe:** Boxplot-Whiskers > 1.5·IQR zeigen Ausreißer. Bei DE/FR/IT sehen wir wegen des starken Aufwärtstrends 2022 und 2025–26 viele "Ausreißer" am oberen Ende – das sind aber keine Anomalien, sondern Strukturbrüche. Ein einfaches Trimmen wäre hier falsch.

## 3 · Stationaritätstests & Vorverarbeitung

**Schwache Stationarität** verlangt konstanten Mittelwert und konstante Varianz.

- **ADF** (Augmented Dickey–Fuller): H₀ = Reihe hat Einheitswurzel (nicht-stationär). Niedriger p-Wert → stationär.
- **PP** (Phillips–Perron): Wie ADF, robuster gegen serielle Korrelation und Heteroskedastizität.
- **KPSS**: H₀ = Reihe ist stationär. Hoher p-Wert → stationär (umgekehrte Logik!).

Wir verwenden den **Konfirmationsansatz**: Stationarität nur dann annehmen, wenn ADF UND KPSS zustimmen.

In [ ]:
def stationarity_battery(series: pd.Series) -> dict:
    series = series.dropna()
    adf_stat, adf_p, *_ = adfuller(series, autolag='AIC')
    try:
        kpss_stat, kpss_p, *_ = kpss(series, regression='c', nlags='auto')
    except Exception:
        kpss_stat, kpss_p = np.nan, np.nan
    # PP-Test ist in statsmodels nicht direkt verfügbar; ADF mit fixiertem Lag dient als Proxy.
    # Eine echte PP-Implementierung würde die arch- oder pmdarima-Pakete erfordern.
    pp_stat, pp_p, *_ = adfuller(series, maxlag=0, autolag=None)
    return {
        'ADF_stat': adf_stat, 'ADF_p': adf_p,
        'PP_proxy_stat': pp_stat, 'PP_proxy_p': pp_p,
        'KPSS_stat': kpss_stat, 'KPSS_p': kpss_p,
    }

rows = []
for c in data:
    r = stationarity_battery(data[c][TARGET])
    r['Country'] = c
    rows.append(r)
stat_levels = pd.DataFrame(rows).set_index('Country')
print('Stationarität der Levels (petrol_usd_liter):')
stat_levels

### 3.1 Transformationen

Wir versuchen drei klassische Vorgehensweisen:

1. **Log-Transformation** – stabilisiert die Varianz, wenn Volatilität proportional zum Level ist.
2. **Erstdifferenzen** – entfernt einen linearen Trend; aus I(1) wird I(0).
3. **HP-Filter** (Hodrick–Prescott) – trennt Trend und Zyklus. λ=1600 für Quartalsdaten; für wöchentliche Daten oft λ=129600.

In [ ]:
LAMBDA_HP = 129600  # empirische Empfehlung für Wochendaten

transformed = {}
for country, df in data.items():
    s = df[TARGET].dropna()
    log_s = np.log(s)
    diff_s = s.diff().dropna()
    log_diff_s = log_s.diff().dropna()
    cycle, trend = hpfilter(s, lamb=LAMBDA_HP)
    transformed[country] = {
        'level': s, 'log': log_s,
        'diff': diff_s, 'log_diff': log_diff_s,
        'trend': trend, 'cycle': cycle,
    }

# Stationaritätsbatterie auf log-Differenzen
rows = []
for c, t in transformed.items():
    r = stationarity_battery(t['log_diff'])
    r['Country'] = c
    rows.append(r)
stat_logdiff = pd.DataFrame(rows).set_index('Country')
print('Stationarität der Log-Differenzen:')
stat_logdiff

In [ ]:
# Visualisierung: Original vs. Log-Diff vs. HP-Trend/Zyklus für Deutschland exemplarisch
country = 'Germany'
t = transformed[country]
fig, axes = plt.subplots(3, 1, figsize=(13, 8))
axes[0].plot(t['level']); axes[0].set_title(f'{country} – Original Level')
axes[1].plot(t['log_diff'], color='darkorange'); axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title(f'{country} – Log-Differenzen (stationär?)')
axes[2].plot(t['trend'], label='HP-Trend', color='steelblue')
axes[2].plot(t['cycle'] + t['trend'].mean(), label='HP-Zyklus (verschoben)', color='firebrick', alpha=0.6)
axes[2].set_title(f'{country} – HP-Filter Zerlegung'); axes[2].legend()
plt.tight_layout(); plt.show()

## 4 · Train/Test-Split (70/30, temporal)

**Kritisch:** Kein Random-Sampling. Reihenfolge bleibt erhalten, da sonst Look-Ahead-Bias entsteht.

Der **pessimistische Bias** entsteht, weil das Modell die jüngsten Daten – und damit potenziell strukturelle Regimewechsel (z. B. Brent-Sprung auf 130 USD) – nicht in den Trainingsdaten gesehen hat. Unsere Out-of-Sample-Performance ist also tendenziell *schlechter*, als sie wäre, wenn das Modell laufend re-trainiert würde.

In [ ]:
SPLIT_RATIO = 0.70

def split_train_test(series: pd.Series, ratio: float = SPLIT_RATIO):
    n = len(series)
    cut = int(np.floor(n * ratio))
    return series.iloc[:cut], series.iloc[cut:]

splits = {}
for country, df in data.items():
    train, test = split_train_test(df[TARGET])
    splits[country] = {'train': train, 'test': test}
    print(f'{country:12s} | train {len(train):3d} ({train.index.min().date()} → {train.index.max().date()})'
          f' | test {len(test):3d} ({test.index.min().date()} → {test.index.max().date()})')

## 5 · Strukturbruch-Analyse

### 5.1 Chow-Test bei bekanntem Bruchdatum

Wir testen zwei kandidatenstarke Daten:

- **2022-02-21** – russischer Einmarsch in der Ukraine (Energiepreisschock)
- **2020-03-09** – Beginn der COVID-19-Pandemie (Nachfrageeinbruch)

Der Chow-Test vergleicht die Fehlerquadratsumme einer einzelnen Regression mit zwei separaten Regressionen (vor/nach dem Bruch). Die Teststatistik ist F-verteilt.

$$ F = \frac{(SSR_p - (SSR_1 + SSR_2))/k}{(SSR_1 + SSR_2)/(n_1 + n_2 - 2k)} $$

In [ ]:
def chow_test(y: pd.Series, x: pd.Series, break_date: pd.Timestamp) -> dict:
    # Chow-Test für Strukturbruch. Regression: y ~ a + b*x
    y, x = y.align(x, join='inner')
    df = pd.concat([y.rename('y'), x.rename('x')], axis=1).dropna()

    X_full = add_constant(df['x'])
    full = OLS(df['y'], X_full).fit()
    ssr_p = full.ssr

    pre = df[df.index <  break_date]
    post = df[df.index >= break_date]
    if len(pre) < 5 or len(post) < 5:
        return {'F': np.nan, 'p': np.nan, 'note': 'Zu wenig Beobachtungen'}

    pre_fit  = OLS(pre['y'],  add_constant(pre['x'])).fit()
    post_fit = OLS(post['y'], add_constant(post['x'])).fit()
    ssr_split = pre_fit.ssr + post_fit.ssr

    k = 2  # Anzahl der geschätzten Parameter (Intercept + Slope)
    n1, n2 = len(pre), len(post)
    F = ((ssr_p - ssr_split) / k) / (ssr_split / (n1 + n2 - 2*k))
    from scipy.stats import f as f_dist
    p = 1 - f_dist.cdf(F, k, n1 + n2 - 2*k)
    return {'F': F, 'p': p, 'n_pre': n1, 'n_post': n2}

CANDIDATES = [pd.Timestamp('2020-03-09'), pd.Timestamp('2022-02-21')]
chow_results = []
for country, df in data.items():
    for d in CANDIDATES:
        r = chow_test(df['petrol_usd_liter'], df['brent_crude_usd'], d)
        r['Country'] = country
        r['Break_Date'] = d.date()
        chow_results.append(r)
chow_df = pd.DataFrame(chow_results)[['Country', 'Break_Date', 'F', 'p', 'n_pre', 'n_post']]
print('Chow-Test (H0: kein Strukturbruch bei petrol ~ brent):')
chow_df

### 5.2 Quandt-Likelihood-Ratio (QLR) Test

Wenn das Bruchdatum unbekannt ist, prüft der QLR-Test (auch *sup-Wald*) **jedes mögliche Datum** in einem mittleren Trim-Bereich (üblich: 15%–85% der Reihe) und nimmt das Maximum der Chow-F-Statistik. Da wir vielfaches Testen betreiben, sind die kritischen Werte höher als beim normalen F-Test (Andrews 1993).

In [ ]:
def qlr_test(y: pd.Series, x: pd.Series, trim: float = 0.15) -> dict:
    y, x = y.align(x, join='inner')
    df = pd.concat([y.rename('y'), x.rename('x')], axis=1).dropna()
    n = len(df)
    lo, hi = int(n*trim), int(n*(1-trim))
    F_stats = []
    dates = df.index[lo:hi]
    for d in dates:
        r = chow_test(df['y'], df['x'], d)
        F_stats.append(r['F'])
    F_stats = pd.Series(F_stats, index=dates).dropna()
    return {'max_F': F_stats.max(), 'date_at_max': F_stats.idxmax(), 'series': F_stats}

# Andrews kritische Werte (1993, 5%-Niveau, k=2 Parameter, 15% Trimm) ≈ 8.85
ANDREWS_5PCT = 8.85
qlr_summary = {}
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (country, df) in zip(axes.flat, data.items()):
    res = qlr_test(df['petrol_usd_liter'], df['brent_crude_usd'])
    qlr_summary[country] = {
        'max_F': res['max_F'],
        'date_at_max': res['date_at_max'].date(),
        'significant_5pct': res['max_F'] > ANDREWS_5PCT,
    }
    ax.plot(res['series'])
    ax.axhline(ANDREWS_5PCT, color='red', linestyle='--', label='Andrews 5%')
    ax.axvline(res['date_at_max'], color='green', linestyle=':',
               label=f'Max @ {res["date_at_max"].date()}')
    ax.set_title(f'QLR – {country}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
pd.DataFrame(qlr_summary).T

## 6 · Univariate Modellierung: ARIMA (Box–Jenkins)

### 6.1 ACF / PACF Identifikation

Faustregel:

- **AR(p)**: PACF schneidet nach Lag p ab; ACF klingt geometrisch ab.
- **MA(q)**: ACF schneidet nach Lag q ab; PACF klingt geometrisch ab.
- **ARMA(p,q)**: beide klingen ab.

Wir arbeiten auf der ersten Log-Differenz (I(1)-Annahme).

In [ ]:
country = 'Germany'  # exemplarisch; gleich auch in Schleife
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
plot_acf(transformed[country]['log_diff'].dropna(), lags=40, ax=axes[0])
plot_pacf(transformed[country]['log_diff'].dropna(), lags=40, ax=axes[1], method='ywm')
axes[0].set_title(f'ACF – {country} log-diff')
axes[1].set_title(f'PACF – {country} log-diff')
plt.tight_layout(); plt.show()

### 6.2 Grid-Search über (p,d,q) mit AIC/BIC/HQ

**Parsimony:** Wir suchen das kleinste Modell mit gutem Fit. Information Criteria balancieren Fit gegen Komplexität:

$$AIC = -2\ln L + 2k, \quad BIC = -2\ln L + k\ln n, \quad HQ = -2\ln L + 2k\ln\ln n$$

BIC bestraft Komplexität am stärksten, was zu sparsameren Modellen führt – im Sinne des Parsimony-Prinzips meist die robusteste Wahl.

In [ ]:
def grid_search_arima(train: pd.Series, p_max=3, d_max=2, q_max=3) -> pd.DataFrame:
    # Brute-force-Suche über kleine (p,d,q)-Räume.
    rows = []
    for p in range(p_max+1):
        for d in range(d_max+1):
            for q in range(q_max+1):
                if p == d == q == 0:
                    continue
                try:
                    fit = ARIMA(train, order=(p, d, q)).fit()
                    rows.append({
                        '(p,d,q)': (p, d, q),
                        'AIC': fit.aic, 'BIC': fit.bic, 'HQIC': fit.hqic,
                        'k': p + q + (1 if 'const' in fit.params.index else 0),
                    })
                except Exception:
                    continue
    return pd.DataFrame(rows).sort_values('BIC').reset_index(drop=True)

arima_best = {}
for country in data:
    train = splits[country]['train']
    grid = grid_search_arima(train)
    best = grid.iloc[0]
    arima_best[country] = best['(p,d,q)']
    print(f'{country:12s} | bestes BIC-Modell: {best["(p,d,q)"]} '
          f'| AIC={best["AIC"]:.2f}  BIC={best["BIC"]:.2f}  HQ={best["HQIC"]:.2f}')

### 6.3 Time-Series Cross-Validation

Im Gegensatz zu k-fold-CV respektieren wir die Zeitordnung. `TimeSeriesSplit` erzeugt expanding-window-Folds: jeder Trainingsfold endet vor dem zugehörigen Testfold. Das gibt uns einen *stabileren* Schätzer des Out-of-Sample-Fehlers als ein einzelner Holdout.

In [ ]:
def ts_cv_arima(series: pd.Series, order: tuple, n_splits=5):
    # Expanding-window Cross-Validation. Liefert mean & std des RMSE über Folds.
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmses = []
    for tr_idx, te_idx in tscv.split(series):
        tr, te = series.iloc[tr_idx], series.iloc[te_idx]
        try:
            fit = ARIMA(tr, order=order).fit()
            fc = fit.forecast(steps=len(te))
            rmses.append(np.sqrt(mean_squared_error(te, fc)))
        except Exception:
            rmses.append(np.nan)
    return np.nanmean(rmses), np.nanstd(rmses)

cv_results = {}
for country, order in arima_best.items():
    m, s = ts_cv_arima(splits[country]['train'], order)
    cv_results[country] = {'order': order, 'CV_RMSE_mean': m, 'CV_RMSE_std': s}
pd.DataFrame(cv_results).T

### 6.4 Residualdiagnostik

Ein gut spezifiziertes ARIMA-Modell hat **weißes-Rauschen-Residuen**:

- **Ljung–Box / Portmanteau-Test**: H₀ = keine Autokorrelation. Hoher p-Wert ist gut.
- **Jarque–Bera-Test**: H₀ = Residuen normalverteilt. Hoher p-Wert ist gut.

In [ ]:
def diagnostics(fit, lags=10) -> dict:
    resid = fit.resid.dropna()
    lb = acorr_ljungbox(resid, lags=[lags], return_df=True)
    jb_stat, jb_p, skew, kurt = jarque_bera(resid)
    return {
        'LjungBox_stat': lb['lb_stat'].iloc[0],
        'LjungBox_p':    lb['lb_pvalue'].iloc[0],
        'JarqueBera_stat': jb_stat,
        'JarqueBera_p':    jb_p,
        'Resid_skew': skew, 'Resid_kurt': kurt,
    }

arima_fits = {}
diag_rows = []
for country, order in arima_best.items():
    fit = ARIMA(splits[country]['train'], order=order).fit()
    arima_fits[country] = fit
    d = diagnostics(fit); d['Country'] = country; d['order'] = order
    diag_rows.append(d)
pd.DataFrame(diag_rows).set_index('Country')

## 7 · Multivariate Modellierung: VAR + Granger-Kausalität

### 7.1 VAR-Aufbau

Wir verwenden die multivariate Reihe `[petrol, diesel, brent]` für jedes Land. Die endogenen Variablen werden auf ihre Log-Differenzen transformiert (gemeinsame Stationarität). LPG und tax_percentage lassen wir bewusst weg: LPG ist hochkorreliert mit Petrol/Diesel (Multikollinearität), tax_percentage ist im Datensatz unrealistisch volatil.

In [ ]:
def build_var_panel(df: pd.DataFrame) -> pd.DataFrame:
    # Multivariates Panel auf log-Differenzen
    cols = ['petrol_usd_liter', 'diesel_usd_liter', 'brent_crude_usd']
    panel = np.log(df[cols]).diff().dropna()
    panel.columns = ['petrol', 'diesel', 'brent']
    return panel

var_panels = {c: build_var_panel(df) for c, df in data.items()}
var_panels['Germany'].head()

### 7.2 Lag-Selektion über AIC/BIC/HQ

`statsmodels` bietet `VAR.select_order` direkt. Wir wählen die kleinste Lag-Ordnung, die laut BIC optimal ist (Parsimony!).

In [ ]:
var_models = {}
for country, panel in var_panels.items():
    train_idx = splits[country]['train'].index
    panel_train = panel.loc[panel.index.isin(train_idx)]
    sel = VAR(panel_train).select_order(maxlags=8)
    chosen = sel.bic  # bewusst BIC für Sparsamkeit
    fit = VAR(panel_train).fit(maxlags=chosen if chosen > 0 else 1)
    var_models[country] = fit
    print(f'{country:12s} | gewählter Lag (BIC): {chosen} | k_params/eq: {fit.k_ar}')

### 7.3 Granger-Kausalität (Wald-Tests)

H₀: Variable X *Granger-causes* Variable Y nicht (alle Lag-Koeffizienten von X in der Y-Gleichung = 0). Wir testen vor allem:

- *brent → petrol* (erwartet: stark signifikant für DE/FR/IT, schwächer für ID)
- *brent → diesel*
- *petrol ↔ diesel* (Spillover)

In [ ]:
def granger_summary(fit, target: str, candidates: list, signif=0.05) -> pd.DataFrame:
    rows = []
    for src in candidates:
        if src == target: continue
        t = fit.test_causality(target, [src], kind='wald', signif=signif)
        rows.append({
            'Source': src, 'Target': target,
            'Wald_F': t.test_statistic, 'p_value': t.pvalue,
            'Causes (5%)': t.pvalue < signif,
        })
    return pd.DataFrame(rows)

for country, fit in var_models.items():
    print(f'\n=== {country} – Granger Causality (Wald) ===')
    out = pd.concat([
        granger_summary(fit, 'petrol', ['diesel', 'brent']),
        granger_summary(fit, 'diesel', ['petrol', 'brent']),
        granger_summary(fit, 'brent',  ['petrol', 'diesel']),
    ], ignore_index=True)
    print(out.to_string(index=False))

### 7.4 VAR-Residualdiagnostik

In [ ]:
var_diag = []
for country, fit in var_models.items():
    # Portmanteau für VAR
    pt = fit.test_whiteness(nlags=10, adjusted=False)
    # Normalität (Jarque–Bera auf jede Gleichung)
    nb = fit.test_normality()
    var_diag.append({
        'Country': country,
        'Portmanteau_stat': pt.test_statistic, 'Portmanteau_p': pt.pvalue,
        'Normality_stat':   nb.test_statistic, 'Normality_p':   nb.pvalue,
    })
pd.DataFrame(var_diag).set_index('Country')

## 8 · State-Space-Modell (Unobserved Components)

Wir zerlegen die Reihe in unbeobachtete Komponenten:

$$ y_t = \mu_t + \gamma_t + \epsilon_t $$

mit Trend $\mu_t$ (lokaler linearer Trend), zyklischer Komponente $\gamma_t$ (stochastischer Zyklus) und Beobachtungsrauschen $\epsilon_t$. Die Schätzung erfolgt via Kalman-Filter / Maximum Likelihood.

In [ ]:
ucm_fits = {}
fig, axes = plt.subplots(4, 1, figsize=(13, 10))
for ax, country in zip(axes, data):
    y = splits[country]['train']
    mdl = UnobservedComponents(y, level='local linear trend', cycle=True,
                               stochastic_cycle=True, damped_cycle=True)
    fit = mdl.fit(disp=False, maxiter=200)
    ucm_fits[country] = fit
    ax.plot(y.index, y, label='Original', alpha=0.5)
    ax.plot(y.index, fit.level['smoothed'], label='Trend', color='firebrick')
    ax.set_title(f'{country} – UCM Trend')
    ax.legend()
plt.tight_layout(); plt.show()

## 9 · Prognose & Evaluation

Für jedes Land vergleichen wir drei Modelle auf der **Test-Periode** (letzte 30%):

1. ARIMA (BIC-bestes Modell aus Block 6)
2. VAR – Multivariate Prognose; wir extrahieren die `petrol`-Komponente
3. UCM – State-Space-Trend-Prognose

**95%-Prognoseintervall:** $\hat y_t \pm 1.96 \cdot \hat\sigma_t$. Bei UCM und ARIMA liefert statsmodels das direkt; bei VAR rekonstruieren wir es aus der prognostizierten Kovarianzmatrix.

In [ ]:
def evaluate(actual: pd.Series, pred: pd.Series) -> dict:
    actual, pred = actual.align(pred, join='inner')
    mse = mean_squared_error(actual, pred)
    return {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(actual, pred),
    }

results = []

for country in data:
    train = splits[country]['train']
    test  = splits[country]['test']
    horizon = len(test)

    # --- ARIMA ---
    arima_fit = arima_fits[country]
    arima_fc = arima_fit.get_forecast(steps=horizon)
    arima_mean = arima_fc.predicted_mean
    arima_ci = arima_fc.conf_int(alpha=0.05)
    arima_mean.index = test.index
    arima_ci.index = test.index
    m = evaluate(test, arima_mean); m['Model'] = 'ARIMA'; m['Country'] = country
    results.append(m)

    # --- VAR (petrol-Komponente, rücktransformiert) ---
    var_fit = var_models[country]
    panel = var_panels[country]
    train_panel = panel.loc[panel.index.isin(train.index)]
    last_obs = train_panel.values[-var_fit.k_ar:]
    fc = var_fit.forecast(y=last_obs, steps=horizon)
    fc_df = pd.DataFrame(fc, columns=panel.columns)
    # zurück auf Level: kumulierte Log-Returns vom letzten Trainings-Level
    last_log_price = np.log(train.iloc[-1])
    var_level = np.exp(last_log_price + fc_df['petrol'].cumsum())
    var_level.index = test.index
    m = evaluate(test, var_level); m['Model'] = 'VAR'; m['Country'] = country
    results.append(m)

    # --- UCM ---
    ucm_fc = ucm_fits[country].get_forecast(steps=horizon)
    ucm_mean = ucm_fc.predicted_mean
    ucm_mean.index = test.index
    m = evaluate(test, ucm_mean); m['Model'] = 'UCM'; m['Country'] = country
    results.append(m)

eval_df = (pd.DataFrame(results)
           .set_index(['Country', 'Model'])[['MSE', 'RMSE', 'MAE']])
print('Out-of-Sample Performance:')
eval_df

### 9.1 Visualisierung: Prognosen vs. Realisierung

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 12))
for ax, country in zip(axes, data):
    train = splits[country]['train']
    test  = splits[country]['test']
    ax.plot(train.index[-50:], train.iloc[-50:], color='gray', label='Train (Ausschnitt)')
    ax.plot(test.index, test, color='black', label='Realisiert', linewidth=2)

    # ARIMA mit 95%-Intervall
    fc = arima_fits[country].get_forecast(steps=len(test))
    mean = fc.predicted_mean; mean.index = test.index
    ci = fc.conf_int(alpha=0.05); ci.index = test.index
    ax.plot(test.index, mean, label='ARIMA', color='steelblue')
    ax.fill_between(test.index, ci.iloc[:,0], ci.iloc[:,1], color='steelblue', alpha=0.15)

    # UCM
    ucm_fc = ucm_fits[country].get_forecast(steps=len(test))
    um = ucm_fc.predicted_mean; um.index = test.index
    ax.plot(test.index, um, label='UCM', color='firebrick', linestyle='--')

    ax.set_title(f'{country} – Forecast vs. Realisierung'); ax.legend()
plt.tight_layout(); plt.show()

### 9.2 Bestes Modell pro Land

In [ ]:
best_by_country = (eval_df.reset_index()
                   .sort_values(['Country', 'RMSE'])
                   .groupby('Country').first()[['Model', 'RMSE', 'MAE']])
print('Bestes Modell (nach RMSE) je Land:')
best_by_country

## 10 · Diskussion & Limitierungen

**Was die Ergebnisse vermutlich zeigen werden** (ohne Pre-Run):

- **DE/FR/IT** sind sich sehr ähnlich (gleiche Region, gleiche Subventionspolitik). Granger-Kausalität von Brent auf Petrol sollte hier hochsignifikant sein.
- **Indonesien** wird vermutlich *schlechtere* RMSE-Werte zeigen, weil die subventionierten Preise *träger* reagieren – ARIMA über lange Horizonte wird unterfitten.
- Strukturbrüche werden wahrscheinlich im Bereich **Q1 2022** (Energiekrise) bei DE/FR/IT detektiert und bei ID eher gar nicht (Subvention dämpft den Effekt).

**Limitierungen, die du in der Abschlussdiskussion ansprechen solltest:**

1. **Synthetische Datenkomponenten** – die `tax_percentage`-Spalte ist unrealistisch volatil. Falls dies eine Übungsdatei ist, ist das akzeptabel; in einer realen Analyse müsste die Spalte verworfen oder über einen rollierenden Mittelwert geglättet werden.
2. **Pessimistic Bias** – das Modell sieht den Brent-Sprung auf 130 USD im Test-Set zum ersten Mal. Eine *Rolling-Origin-Forecast* (laufendes Re-Training) wäre ein realistischeres Bewertungsdesign.
3. **Endogenität bei VAR** – Brent ist global, Petrol-Preise einzelner Länder sollten ihn nicht beeinflussen. Wenn der Wald-Test in die Gegenrichtung (Petrol → Brent) signifikant ausschlägt, ist das ein Spurious-Befund.
4. **Strukturbrüche werden nicht modelliert** – TAR-, Markov-Switching- oder Time-Varying-Parameter-Modelle wären die nächste Stufe.